## Script to convert a YOLO dataset (like Zenodo 7395752 dataset) to a JSON file compatible with COCO format

In [ ]:
import pandas as pd
import json
import os
import cv2
import numpy as np
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
import random

DATASETS_PATH = os.environ.get("ZERO_SHOT_DATA_DIR", "./data")
ANNOTATIONS_DIR = os.path.join(DATASETS_PATH, "7395752/train1201")  # folder with .txt YOLO annotation files
IMAGES_DIR = ANNOTATIONS_DIR  # folder with image files
OUTPUT_JSON = ANNOTATIONS_DIR + "_coco.json"  # output COCO JSON file

ORIGINAL_CATEGORIES = { # id: name mapping from the original dataset
    0: "Coccinellidae septempunctata",
    1: "Apis mellifera",
    2: "Bombus lapidarius",
    3: "Bombus terrestris",
    4: "Eupeodes corolla",
    5: "Episyrphus balteatus",
    6: "Aglais urticae",
    7: "Vespula vulgaris",
    8: "Eristalis tenax",
    10: "Bombus spp.",
    11: "Syrphidae"
}

def yolo_to_coco(annotations_dir, images_dir, categories, output_json):
    coco_output = {
        "info": {
            "description": "Converted from YOLO Format",
            "year": datetime.now().year,
            "date_created": datetime.now().strftime("%Y-%m-%d")
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": [
            {"id": cat_id, "name": cat_name, "supercategory": "insect"}
            for cat_id, cat_name in categories.items()
        ]
    }

    current_img_id = 1
    current_ann_id = 1

    annotation_files = [f for f in os.listdir(annotations_dir) if f.endswith('.txt')]
    print(f"Processing {len(annotation_files)} annotation files...")

    for ann_file in tqdm(annotation_files):
        base_name = os.path.splitext(ann_file)[0]
        
        # Find corresponding image
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
            potential_path = os.path.join(images_dir, base_name + ext)
            if os.path.exists(potential_path):
                img_path = potential_path
                break
        
        if img_path is None:
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue
        
        height, width = img.shape[:2]
        
        image_info = {
            "id": current_img_id,
            "file_name": os.path.basename(img_path),
            "width": width,
            "height": height
        }
        coco_output["images"].append(image_info)

        # Read YOLO annotations
        ann_path = os.path.join(annotations_dir, ann_file)
        with open(ann_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                
                class_id = int(parts[0])
                x_center = float(parts[1]) * width
                y_center = float(parts[2]) * height
                box_width = float(parts[3]) * width
                box_height = float(parts[4]) * height
                
                # Convert to COCO format (x, y, width, height) where x,y is top-left
                x_min = x_center - box_width / 2
                y_min = y_center - box_height / 2
                
                annotation = {
                    "id": current_ann_id,
                    "image_id": current_img_id,
                    "category_id": class_id,
                    "segmentation": [],
                    "area": box_width * box_height,
                    "bbox": [x_min, y_min, box_width, box_height],
                    "iscrowd": 0
                }
                
                coco_output["annotations"].append(annotation)
                current_ann_id += 1
        
        current_img_id += 1

    with open(output_json, 'w') as f:
        json.dump(coco_output, f, indent=4)
    
    print(f"Images: {len(coco_output['images'])}")
    print(f"Annotations: {len(coco_output['annotations'])}")
    print(f"Saved to: {output_json}")

def visualize_category_distribution(coco_json):
    with open(coco_json, 'r') as f:
        coco_data = json.load(f)
    
    category_counts = {cat['id']: 0 for cat in coco_data['categories']}
    for annotation in coco_data['annotations']:
        category_counts[annotation['category_id']] += 1
    
    category_names = [cat['name'] for cat in coco_data['categories']]
    counts = [category_counts[cat['id']] for cat in coco_data['categories']]

    print("Background images (no annotations):", len(coco_data['images']) - len(set(ann['image_id'] for ann in coco_data['annotations'])))
    
    plt.figure(figsize=(10, 5))
    bars = plt.bar(category_names, counts)
    
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom')
    
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Category')
    plt.ylabel('Count')
    plt.title('Category Distribution')
    plt.tight_layout()
    plt.show()

def visualize_random_samples(coco_json, images_dir, num_samples=5):
    with open(coco_json, 'r') as f:
        coco_data = json.load(f)
    
    images_info = {img['id']: img for img in coco_data['images']}
    annotations_by_image = {}
    for ann in coco_data['annotations']:
        img_id = ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        annotations_by_image[img_id].append(ann)
    
    sample_images = random.sample(list(images_info.keys()), num_samples)
    
    for img_id in sample_images:
        img_info = images_info[img_id]
        img_path = os.path.join(images_dir, img_info['file_name'])
        img = cv2.imread(img_path)
        
        if img is None:
            continue
        
        anns = annotations_by_image.get(img_id, [])
        for ann in anns:
            x, y, w, h = ann['bbox']
            category_id = ann['category_id']
            category_name = next((cat['name'] for cat in coco_data['categories'] if cat['id'] == category_id), "Unknown")
            cv2.rectangle(img, (int(x), int(y)), (int(x+w), int(y+h)), (0, 0, 255), 2)
            cv2.putText(img, category_name, (int(x), int(y)-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        
        plt.figure(figsize=(8, 8))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(f"Image ID: {img_id}")
        plt.show()

In [ ]:
yolo_to_coco(ANNOTATIONS_DIR, IMAGES_DIR, ORIGINAL_CATEGORIES, OUTPUT_JSON)

In [ ]:
visualize_random_samples(OUTPUT_JSON, IMAGES_DIR)

In [ ]:
main_mapping = {
    0: "Coleoptera",
    1: "Diptera-Hymenoptera",
    2: "Lepidoptera",
    3: "Araneae",
    4: "Hemiptera", # only on ArTaxOr, not in pitfall dataset
    5: "Odonata",   # only on ArTaxOr, not in pitfall dataset
    6: "Dermaptera",
    7: "Myriapoda",
    8: "Isopoda",
    9: "Opiliones"
}

unified_id_mapping = {
    # COLEOPTERA
    "carabidae": 0,
    "staphylinidae": 0,
    "coccinellidae": 0,
    "cantharidae": 0,
    "coleoptera (larvae)": 0,
    "Coccinellidae septempunctata": 0,
    "Coleoptera": 0,
    
    # DIPTERA-HYMENOPTERA
    "diptera-hymenoptera": 1,
    "Apis mellifera": 1,
    "Bombus lapidarius": 1,
    "Bombus terrestris": 1,
    "Vespula vulgaris": 1,
    "Eupeodes corolla": 1,
    "Episyrphus balteatus": 1,
    "Eristalis tenax": 1,
    "Diptera": 1,
    "Hymenoptera": 1,
    "Bombus spp.": 1,
    "Syrphidae": 1,

    # LEPIDOPTERA
    "Aglais urticae": 2,
    "Lepidoptera": 2,

    # ARANEAE
    "araneae": 3,
    "Araneae": 3,

    "Hemiptera": 4,
    "Odonata": 5,
    "dermaptera": 6,
    "myriapoda": 7,
    "isopoda": 8,
    "opiliones": 9,
}

In [ ]:
def convert_coco_categories(input_json, output_json, id_mapping, new_categories):
    with open(input_json, 'r') as f:
        coco_data = json.load(f)
    
    original_categories = {cat['id']: cat['name'] for cat in coco_data['categories']}
    
    new_annotations = []
    images_to_skip = set()
    
    for annotation in coco_data['annotations']:
        category_id = annotation['category_id']
        
        original_category_name = original_categories.get(category_id)
        
        if original_category_name is None:
            print(f"Warning: No category found for id {category_id}, skipping annotation {annotation['id']}")
            images_to_skip.add(annotation['image_id'])
            continue

        new_category_id = id_mapping.get(original_category_name)
        if new_category_id is None:
            print(f"Warning: No mapping found for category '{original_category_name}' (id: {category_id}), skipping annotation {annotation['id']}")
            images_to_skip.add(annotation['image_id'])
            continue
        
        annotation['category_id'] = new_category_id
        new_annotations.append(annotation)
    
    new_annotations = [ann for ann in new_annotations if ann['image_id'] not in images_to_skip]
    
    filtered_images = [img for img in coco_data['images'] if img['id'] not in images_to_skip]
    
    coco_data['images'] = filtered_images
    coco_data['annotations'] = new_annotations
    
    coco_data['categories'] = [
        {"id": cat_id, "name": cat_name, "supercategory": "insect"}
        for cat_id, cat_name in new_categories.items()
    ]

    with open(output_json, 'w') as f:
        json.dump(coco_data, f, indent=4)
    
    images_with_annotations = len(set(ann['image_id'] for ann in new_annotations))
    background_images = len(coco_data['images']) - images_with_annotations
    
    print(f"\nConverted categories in {output_json}")
    print(f"Total images: {len(coco_data['images'])} (annotated: {images_with_annotations}, background: {background_images})")
    print(f"Total annotations: {len(new_annotations)}")
    print(f"Skipped images with invalid annotations: {len(images_to_skip)}")

convert_coco_categories(OUTPUT_JSON, OUTPUT_JSON.replace(".json", "_unified.json"), unified_id_mapping, main_mapping)


In [ ]:
visualize_category_distribution(OUTPUT_JSON.replace(".json", "_unified.json"))

In [ ]:
visualize_random_samples(OUTPUT_JSON.replace(".json", "_unified.json"), IMAGES_DIR, num_samples=1)

In [ ]:
import json
import os
import cv2
import numpy as np
from matplotlib import pyplot as plt
from pycocotools import mask as mask_util
from collections import defaultdict

def plot_annotations_with_crops(coco_json, img_dir, num_images=5, max_crops_per_img=10):
    with open(coco_json) as f:
        coco_data = json.load(f)

    annotated_img_ids = {ann['image_id'] for ann in coco_data['annotations']} # if ann['category_id'] == 2}
    images = random.sample([img for img in coco_data['images'] if img['id'] in annotated_img_ids], min(num_images, len([img for img in coco_data['images'] if img['id'] in annotated_img_ids])))

    img_to_anns = defaultdict(list)
    for ann in coco_data['annotations']:
        img_to_anns[ann['image_id']].append(ann)

    print(f"Visualizing {len(images)} images...")

    for img_info in images:
        img_id = img_info['id']
        filename = img_info['file_name']
        
        img_path = os.path.join(img_dir, filename)
        if not os.path.exists(img_path):
            img_path = os.path.join(img_dir, 'images', filename)
        
        full_img_bgr = cv2.imread(img_path)
        if full_img_bgr is None:
            print(f"Error loading {filename}")
            continue
        
        print(f"Loaded image: {filename} (ID: {img_id})")
            
        full_img_rgb = cv2.cvtColor(full_img_bgr, cv2.COLOR_BGR2RGB)
        h, w = full_img_rgb.shape[:2]
        
        anns = img_to_anns[img_id][:max_crops_per_img]
        
        num_crops = len(anns)
        cols = min(5, num_crops)
        rows = 1 + (num_crops + cols - 1) // cols
        
        _, axes = plt.subplots(rows, cols, figsize=(12*cols, 6*rows))
        if rows == 1:
            axes = axes.reshape(1, -1)
        elif cols == 1:
            axes = axes.reshape(-1, 1)
        
        main_view = full_img_rgb.copy()
        for ann in anns:
            mask = _decode_mask(ann['segmentation'], h, w)
            
            if mask is not None:
                contours, _ = cv2.findContours((mask * 255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(main_view, contours, -1, (0, 255, 0), 2)
                
                x, y, bw, bh = map(int, ann['bbox'])
                cv2.rectangle(main_view, (x, y), (x+bw, y+bh), (255, 0, 0), 2)
            else:
                x, y, bw, bh = map(int, ann['bbox'])
                cv2.rectangle(main_view, (x, y), (x+bw, y+bh), (255, 0, 0), 2)
        
        axes[0, 0].imshow(main_view)
        axes[0, 0].set_title(f"{filename} ({len(anns)} annotations)", fontsize=10)

        axes[0, 0].axis('off')
        
        for j in range(1, cols):
            axes[0, j].axis('off')
        
        for i, ann in enumerate(anns):
            row = 1 + i // cols
            col = i % cols
            
            x, y, bw, bh = map(int, ann['bbox'])
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(w, x + bw), min(h, y + bh)
            
            if x2 > x1 and y2 > y1:
                crop = full_img_rgb[y1:y2, x1:x2]
                category = next((cat['name'] for cat in coco_data['categories'] if cat['id'] == ann['category_id']), "Unknown")
                
                axes[row, col].imshow(crop)
                axes[row, col].set_title(f"{category}\nID: {ann['id']}", fontsize=8)
                axes[row, col].axis('off')
            else:
                axes[row, col].text(0.5, 0.5, "Invalid", ha='center', va='center')
                axes[row, col].axis('off')
        
        for i in range(num_crops, rows * cols - cols):
            row = 1 + i // cols
            col = i % cols
            axes[row, col].axis('off')
        
        plt.tight_layout()
        plt.show()

def _decode_mask(segmentation, h, w):
    if segmentation == []:
        return None
    if isinstance(segmentation, dict):
        return mask_util.decode(segmentation)
    elif isinstance(segmentation, list):
        mask = np.zeros((h, w), dtype=np.uint8)
        for seg in segmentation:
            poly = np.array(seg).reshape(-1, 2).astype(np.int32)
            cv2.fillPoly(mask, [poly], 1)
        return mask
    return None


plot_annotations_with_crops(OUTPUT_JSON.replace(".json", "_unified.json"), IMAGES_DIR, num_images=1)
